In [73]:
# ====================================================
# ETL GEOESPACIAL - MUNICÍPIOS DE SÃO PAULO
# Autor: Jessé Lima
# Os casos_registrados são fictícios, apenas servindo de uma maneira a demonstrar competência técnica em ETL usando Python
# ====================================================

import pandas as pd # Manipulação de dados tabulares
import geopandas as gpd # Manipulação de dados geoespaciais
from shapely.validation import make_valid # Correção de geometrias inválidas
import urllib.request #Download de arquivos via URL
import os # Comunicação com sistema operacional

# **E = EXTRACT**

In [74]:
# URL da malha municipal do Estado de São Paulo
url_ibge = (
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/"
    "malhas_territoriais/malhas_municipais/municipio_2022/"
    "UFs/SP/SP_Municipios_2022.zip"
)

# Nome local do arquivo a ser baixado
arquivo_zip = "SP_Municipios_2022.zip"

# Verifica se o arquivo já existe
if not os.path.exists(arquivo_zip):
    print("Baixando malha municipal do IBGE...")
    urllib.request.urlretrieve(
        url_ibge,
        arquivo_zip
    )
    print("Download concluído!") #Se não tiver integrado, realiza o download e informa

else:
    print("Arquivo já existe.") #Caso apareça esse comando, ele já está integrado na máquina do Colab

Arquivo já existe.


In [75]:
# Checando os arquivos presentes que estão integrados (pipeline_sp_saude_otimizado.parquet  sample_data  SP_Municipios_2022.zip)
!ls
''

sample_data  SP_Municipios_2022.zip


''

In [76]:
# Leitura do arquivo vetorial diretamente do ZIP
gdf_sp = gpd.read_file(
    f"zip://{arquivo_zip}",
    engine="pyogrio" # Essa biblioteca otimiza a extração de arquivos vetoriais pesados
)

# Visualização das 5 primeiras linhas contendo
gdf_sp.head()

# Visualização das 5 últimas linhas contendo
gdf_sp.tail() # É possível colocar um número dentro do () para checar mais linhas

# Ambas informam as colunas do nº da linha, CD_MUN, SIGLA_UF, AREA_KM2

,CD_MUN,NM_MUN,SIGLA_UF,AREA_KM2,geometry
640,3557006,Votorantim,SP,184.186,"POLYGON ((-47.42427 -23.5242, -47.42425 -23.52..."
641,3557105,Votuporanga,SP,420.703,"POLYGON ((-49.9832 -20.57103, -49.98408 -20.57..."
642,3557154,Zacarias,SP,319.056,"POLYGON ((-49.98507 -21.05306, -49.98518 -21.0..."
643,3557204,Chavantes,SP,188.727,"POLYGON ((-49.72713 -23.10799, -49.72833 -23.1..."
644,3557303,Estiva Gerbi,SP,74.144,"POLYGON ((-46.96444 -22.29548, -46.9661 -22.29..."


In [77]:
# Base simulada do DATASUS
dados_datasus_sp = {

    "cod_ibge_6": [ # Código dos municípios - IBGE 6 dígitos
        "355030",
        "350950",
        "351880",
        "354850",
        "354340"
    ],

    "MUN_DATASUS": [ # Nome dos municípios com erros no código de programação (espaços, por exemplo)
        " São Paulo ",
        " Campinas ",
        " Guarulhos",
        " Santos ",
        " Ribeirão Preto "
    ],

    "casos_registrados": [ # Dei valor a apenas 3 municípios a caráter exemplificativo de valores presentes, ausentes ou divergentes (string/numero)
        1500, # Exemplo de valor válido
        None, # Indica a ausência de valor
        420,
        85,
        "NaN" # Informa texto no lugar de número
    ]
}

# Transformando o dicionário em uma tabela do Pandas
df_saude = pd.DataFrame(
    dados_datasus_sp
)

df_saude

,cod_ibge_6,MUN_DATASUS,casos_registrados
0,355030,São Paulo,1500
1,350950,Campinas,None
2,351880,Guarulhos,420
3,354850,Santos,85
4,354340,Ribeirão Preto,NaN


# **T = TRANSFORM**



In [78]:
# ====================================================
# TRANSFORMAÇÃO DOS DADOS DATASUS
# ====================================================

df_saude["MUN_DATASUS"] = (
    df_saude["MUN_DATASUS"]
    .str.strip() #Função boa para remover espaços desnecessários no código
)

# Converte a coluna para valor numérico (texto inválido = NaN)
df_saude["casos_registrados"] = (
    pd.to_numeric( # Transforma texto em número
        df_saude["casos_registrados"],
        errors="coerce" # Evita quebrar o código ao encontrar NaN
    )
)

# Substitui valores nulos por 0
df_saude["casos_registrados"] = (
    df_saude["casos_registrados"]
    .fillna(0) # Troca NaN por 0
)

df_saude["casos_registrados"] = (
    df_saude["casos_registrados"]
    .astype(int) # Transforma em número inteiro
)

df_saude

,cod_ibge_6,MUN_DATASUS,casos_registrados
0,355030,São Paulo,1500
1,350950,Campinas,0
2,351880,Guarulhos,420
3,354850,Santos,85
4,354340,Ribeirão Preto,0


In [79]:
gdf_sp["cod_ibge_6"] = (
    gdf_sp["CD_MUN"]
    .astype(str) # tipo: string (texto, caracteres)
    .str[:6] # Delimita como 6 dígitos o string
)

# Une as duas fontes de dados (IBGE e DATASUS) pelo CD_MUN
gdf_integrado = gdf_sp.merge(
    df_saude,
    on="cod_ibge_6",
    how="left"
)

gdf_integrado.head()

,CD_MUN,NM_MUN,SIGLA_UF,AREA_KM2,geometry,cod_ibge_6,MUN_DATASUS,casos_registrados
0,3500105,Adamantina,SP,411.987,"POLYGON ((-51.09557 -21.57029, -51.09617 -21.5...",350010,NaN,NaN
1,3500204,Adolfo,SP,211.055,"POLYGON ((-49.61249 -21.2611, -49.61249 -21.26...",350020,NaN,NaN
2,3500303,Aguaí,SP,474.554,"POLYGON ((-47.01254 -22.00527, -47.01219 -22.0...",350030,NaN,NaN
3,3500402,Águas da Prata,SP,142.673,"POLYGON ((-46.71875 -21.95837, -46.71878 -21.9...",350040,NaN,NaN
4,3500501,Águas de Lindóia,SP,60.126,"POLYGON ((-46.61761 -22.51172, -46.61864 -22.5...",350050,NaN,NaN


In [80]:
# Checando se o merge funcionou
gdf_integrado[
    gdf_integrado["casos_registrados"].notna()
][["MUN_DATASUS", "casos_registrados"]]

,MUN_DATASUS,casos_registrados
108,Campinas,0.0
212,Guarulhos,420.0
487,Ribeirão Preto,0.0
542,Santos,85.0
562,São Paulo,1500.0


In [81]:
# Checando se o join funcionou adequadamente
print(
    gdf_integrado["casos_registrados"]
    .notna()
    .sum()
)

5


In [82]:
gdf_integrado[
    gdf_integrado["casos_registrados"].notna()
]

,CD_MUN,NM_MUN,SIGLA_UF,AREA_KM2,geometry,cod_ibge_6,MUN_DATASUS,casos_registrados
108,3509502,Campinas,SP,794.571,"POLYGON ((-46.99878 -22.93474, -46.99915 -22.9...",350950,Campinas,0.0
212,3518800,Guarulhos,SP,318.675,"POLYGON ((-46.39541 -23.45156, -46.39516 -23.4...",351880,Guarulhos,420.0
487,3543402,Ribeirão Preto,SP,650.916,"POLYGON ((-47.65926 -21.15525, -47.65935 -21.1...",354340,Ribeirão Preto,0.0
542,3548500,Santos,SP,281.033,"MULTIPOLYGON (((-46.3497 -23.97084, -46.34968 ...",354850,Santos,85.0
562,3550308,São Paulo,SP,1521.202,"POLYGON ((-46.63512 -23.8385, -46.63455 -23.84...",355030,São Paulo,1500.0


In [91]:
# Municípios sem registro recebem valor zero
gdf_integrado["casos_registrados"] = (
    gdf_integrado["casos_registrados"]
    .fillna(0) # Filtra os valores Na como 0
    .astype(int) # Define o tipo como valor inteiro
)

# Conferência
gdf_integrado[
    ["NM_MUN", "casos_registrados"]
].head()

,NM_MUN,casos_registrados
0,Adamantina,0
1,Adolfo,0
2,Aguaí,0
3,Águas da Prata,0
4,Águas de Lindóia,0


In [90]:
# Surgiu algum erro nas geometrias, então estou checando se elas foram identificadas adequadamente

print(
    f"Geometrias válidas: {gdf_integrado.is_valid.sum()}"
)

print(
    f"Total de registros: {len(gdf_integrado)}"
)

# Tudo certo até então. 645 de 645 geometrias válidas

Geometrias válidas: 645
Total de registros: 645


In [92]:
gdf_integrado = gdf_integrado.to_crs(
    epsg=4326 # Adequando o sistema de coordenadas ao WGS 84
)

print(
    gdf_integrado.crs
)

EPSG:4326


# **L = LOAD**

In [97]:
# ====================================================
# Exportação GeoParquet
# ====================================================

arq_saida = (
    "etl_saude_municipios_sp.parquet"
)

# Exportando o GeoDataFrame para GeoParquet: a info sai da memória para virar um arquivo
gdf_integrado.to_parquet(
    arq_saida,
    engine="pyarrow"
)

print(
    f"Arquivo gerado: {arq_saida}"
)

Arquivo gerado: etl_saude_municipios_sp.parquet


In [98]:
!ls


etl_saude_municipios_sp.parquet  sample_data  SP_Municipios_2022.zip


In [99]:
# Ler novamente o arquivo gerado

gdf_validacao = gpd.read_parquet(
    arq_saida
)

gdf_validacao.head()

,CD_MUN,NM_MUN,SIGLA_UF,AREA_KM2,geometry,cod_ibge_6,MUN_DATASUS,casos_registrados
0,3500105,Adamantina,SP,411.987,"POLYGON ((-51.09557 -21.57029, -51.09617 -21.5...",350010,None,0
1,3500204,Adolfo,SP,211.055,"POLYGON ((-49.61249 -21.2611, -49.61249 -21.26...",350020,None,0
2,3500303,Aguaí,SP,474.554,"POLYGON ((-47.01254 -22.00527, -47.01219 -22.0...",350030,None,0
3,3500402,Águas da Prata,SP,142.673,"POLYGON ((-46.71875 -21.95837, -46.71878 -21.9...",350040,None,0
4,3500501,Águas de Lindóia,SP,60.126,"POLYGON ((-46.61761 -22.51172, -46.61864 -22.5...",350050,None,0


In [107]:
gdf_validacao[
    gdf_validacao["casos_registrados"] > 0 # Observando os municípios que tiveram registros maiores do que 0
][
    ["NM_MUN", "casos_registrados"] # Retornando a visualização das 2 colunas citadas
]


,NM_MUN,casos_registrados
212,Guarulhos,420
542,Santos,85
562,São Paulo,1500


In [ ]:
# Quantidade de registros
print(len(gdf_teste))

In [102]:
gdf_validacao.geometry.head() # Checando a preservação da geometria

,geometry
0,"POLYGON ((-51.09557 -21.57029, -51.09617 -21.5..."
1,"POLYGON ((-49.61249 -21.2611, -49.61249 -21.26..."
2,"POLYGON ((-47.01254 -22.00527, -47.01219 -22.0..."
3,"POLYGON ((-46.71875 -21.95837, -46.71878 -21.9..."
4,"POLYGON ((-46.61761 -22.51172, -46.61864 -22.5..."


In [103]:
# Etapa final de download do arquivo para checar a sua integridade

from google.colab import files

files.download(arq_saida)

# Arquivo parquet gerado e baixado normalmente

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [109]:
# Para quem quiser, um código para transformar o arquivo em xlsx para download e visualização em excel
gdf_integrado.drop(
    columns="geometry"
).to_excel(
    "etl_saude_municipios_sp.xlsx",
    index=False
)

from google.colab import files

files.download(
    "etl_saude_municipios_sp.xlsx"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Resultado:** Pipeline ETL geoespacial desenvolvido em Python e Google Colab para extração da malha municipal do IBGE, transformação e integração de dados simulados de saúde e geração de um arquivo GeoParquet otimizado para consumo em ferramentas GIS e BI.